# Predicting Reversibility of Photomechanical Molecular Crytals

### Problem Statement

Molecular crystals respond to a light stimulus by diverse kinematic behaviors ranging form reversibly bending and twisting to irrevesibly fragmentating, jumping, and exploding. These behaviors are known to be affected by, among others, the intensity of the incident light, the aspect ratios of crystal geometries, and the volume changes accompanying phase transformation. While these factors, individually, explain the increase in internal energy of the system and its subsequent minimization through macroscopic deformation, they do not fully explain why some crystals revesibly bend and twist while others fragment and jump?

#### Goal

Studies in litrature have focused on individual factors but do not capture the complex interplay of various of these facotrs in governing the photomechanical response. In this project, we aim at predicting whether a crystal undergoes reversible or irrersible photomechanical deformation undergiven conditions such as light intesity, crystal geometry etc.

#### Motivation

The predictive framework could serve as a tool for designing molecular crystalline systems with programmable photomechanical response.

## Import Libraries

In [ ]:
#import libraries here
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from fractions import Fraction
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## Read the data set

In [ ]:
data = pd.read_csv("MolecularCrystalData.csv")

### Understand the data

In [ ]:
print(f"Shape of the dataset: {data.shape}")
data.head() # To visualize top rows on the data

In [ ]:
data.info() # See the information about colums, no of non-null entries and their data type
# data.describe() # Un comment if we are interested in mean, standard deviation, and other metrics about the feature values

We notice that Ratio of symmetry and ref b are by mistake classfied as object, we need to fix this.

In [ ]:
for col in data.columns:

    if data[col].dtype == "object":

        print("\nCOLUMN:", col)

        print(data[col].unique())

In [ ]:
data["ref b"] = (
    data["ref b"]
    .str.replace("`", "", regex=False)
    .astype(float)
)

In [ ]:
data["Ratio of symmetry"] = (
    data["Ratio of symmetry"]
    .apply(lambda x: float(Fraction(str(x))))
)

In [ ]:
data.info() # See the information about colums, no of non-null entries and their data type

In [ ]:
for col in data.columns:

    if data[col].dtype == "object":

        print("\nCOLUMN:", col)

        print(data[col].unique())

### Visualize missing values in the dataset

In [ ]:
missing = data.isnull().sum()
missing_percent = 100 * missing / len(data)

missing_table = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_percent
})

missing_table[missing_table["Missing Count"] > 0]

In [ ]:
# Visualize heat map to see the missing data
plt.figure(figsize=(12,6))

sns.heatmap(
    data.isnull(),
    cbar=False,
    yticklabels=False
)

plt.title("Missing Values Heatmap")
plt.show()

In [ ]:
# Visualize distribution of two target classes
sns.countplot(data=data, x='reversible binary')

plt.title("Distribution of Reversible and Irreversible Class")
plt.show()

In [ ]:
# Number of features with numerical data
numeric_cols = data.select_dtypes(include=np.number).columns
print(f"No of features with numerical values: {len(numeric_cols)}")
print(numeric_cols)

In [ ]:
# understand the distribution of the range of feature data
data[numeric_cols].hist(
    figsize=(16,12),
    bins=15
)

plt.tight_layout()
plt.show()

In [ ]:
# Understand the correlation between the features
# Correlation with the target class can give an intutition before training any data
corr = data[numeric_cols].corr()

plt.figure(figsize=(15,12))

sns.heatmap(
    corr,
    cmap='coolwarm',
    center=0
)

plt.title("Correlation Matrix")
plt.show()

In [ ]:
# Correlation with the target class
corr_target = corr["reversible binary"].sort_values(
    ascending=False
)

print(corr_target)

## Cleaning the data

### 1. Impute the missing values if we think it is appropriate
### 2. Drop columns (features) with missing data
### 3. Drop rows (data points) with missing data

In [ ]:
### Imput the missing Z and Z' values
data_filled = data.copy()

data_filled = data_filled.fillna({
    "Z ratio": 1,
    "Z' ratio": 1,
    #"temperature K": data["temperature K"].median(),
    #"intensity mW/cm^2": data["intensity mW/cm^2"].median()
})

# Visualize heat map to see the missing data
plt.figure(figsize=(12,6))

sns.heatmap(
    data_filled.isnull(),
    cbar=False,
    yticklabels=False
)

plt.title("Missing Values Heatmap")
plt.show()

# save updated data file

data_filled.to_csv(
    "MolecularCrystalData_filledZ.csv",
    index=False
)

### Drop columns with missing data

In [ ]:
cols_with_missing = data_filled.columns[data_filled.isnull().any()] #notice I have imputed data we can use raw data as well

print(cols_with_missing)

In [ ]:
data_drop_cols = data_filled.dropna(axis=1)

print(data_drop_cols.shape)

# data_drop_cols.isnull().sum() # To verify the count of missing data from the features

### Drop rows with missing values

In [ ]:
data_drop_rows = data_filled.dropna()

print(data_drop_rows.shape)
# data_drop_rows.isnull().sum() # To verify the count of missing data from the features

In [ ]:
print("Original:", data.shape)

print("Drop Columns:", data_drop_cols.shape)

print("Drop Rows:", data_drop_rows.shape)

In [ ]:
data_drop_cols.to_csv(
    "MolecularCrystalData_drop_columns.csv",
    index=False
)

data_drop_rows.to_csv(
    "MolecularCrystalData_drop_rows.csv",
    index=False
)

## Train-Test split and define target and features on the cleaned data

In [ ]:
# Choose ONE of the cleaned dataset, without reusing data variable names


# data1 = pd.read_csv("MolecularCrystalData_drop_columns.csv")

# data2 = pd.read_csv("MolecularCrystalData_drop_rows.csv")

#data3 = pd.read_csv("MolecularCrystalData_filledZ.csv")

data0 = data_filled

#drop sample index and "reversible" because those were human bookkeeping collumns
#drop intensity and temp because too much info is missing to use
data0 = data0.drop(
    columns=['Sample Index',
             'reversible',
             'intensity mW/cm^2',
             'temperature K'
            ]
)

print(f"New data shape: {data0.shape}")
data0.head()

In [ ]:
#one data point is missing geometry data (length/width and length/thickness), so we can use a data set without those columns 
data0_nogeometry = data0.dropna(axis=1)
print(f"Data without geometry features shape: {data0_nogeometry.shape}")

#however, we suspect geometry has a major impact on reversibility (can see this from the coorelation data), so we can also use a
#data set that removes the data point missing geometry data
data00 = data0.dropna()
print(f"New data shape: {data00.shape}")

In [ ]:
target = 'reversible binary'

y = data00[target]

X = data00.drop(
    columns=[target]
)

print("X shape:", X.shape)
print("y shape:", y.shape)

### Train-Test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

### Tree model selection and implementation
Random Forest Classifier and Gradient Boosting Classifier

In [ ]:
#initialize random forest classifier - random forest is the second best model for everything right?
rfc = RandomForestClassifier(random_state=42)
rfc.fit(X_train, y_train)
print(f"Random forest training score: {rfc.score(X_train, y_train)}")
print(f"Random forest test score: {rfc.score(X_test, y_test)}")


#initialize gradient boosting classifier to build on random forest model
gbc = GradientBoostingClassifier(random_state=42)
gbc.fit(X_train, y_train)
print(f"\n\nGradient boosting training score: {gbc.score(X_train, y_train)}")
print(f"Gradient boosting test score: {gbc.score(X_test, y_test)}")

In [ ]:
#Let's try cross validation on both models to see which may overall be stronger
#initialize un-trained models again and check cross validated scoring
rfc_cv = RandomForestClassifier(random_state=42)
rfc_scores = cross_val_score(rfc_cv, X, y, cv=5)
print(f"Random forest cross validation scores: {rfc_scores}")
print(f"Random forest average cross validation scores: {rfc_scores.mean()}")


gbc_cv = GradientBoostingClassifier(random_state=42)
gbc_scores = cross_val_score(gbc_cv, X, y, cv=5)
print(f"\n\nGradient Boosting cross validation scores: {gbc_scores}")
print(f"Gradient Boosting average cross validation scores: {gbc_scores.mean()}")

### Feature analysis for random forest model

In [ ]:
rfc_importances = rfc.feature_importances_
feature_names = X.columns.values.tolist()
feature_number = len(feature_names)

fig, ax = plt.subplots()
ax.bar(range(len(rfc_importances)), rfc_importances, color='blue')
ax.set_xlabel('Feature Index Number')
ax.set_ylabel('Normalized Importance')
ax.grid()

In [ ]:
#find most impacting features
list_rfc_i = rfc_importances.tolist()
rank_importances_rfc = np.argsort(list_rfc_i)

#find n most importance features
n=5
print('Most important features to predict reversiblity (highest first):\n')

for i in range(n):
    print(f"{i+1}. {feature_names[rank_importances_rfc[feature_number-1-i]]}")

In [ ]:
#find least impacting features

#find m most importance features
m=5
print('Least important features to predict reversiblity (lowest first):\n')

for i in range(m):
    print(f"{feature_number-i}. {feature_names[rank_importances_rfc[i]]}")

#### Discussion - Explain what these relative feature importances may say about the material system:
The most important features:

"density ratio": The density ratio being important makes sense - if a material is becoming much more compact or much more enlarged, that will cause significant strain on the material. Density ratios far from 1 could make a material more likely to behave irreversibly.
"Abs(DetU-1)" tells you how much the volume changed relative to 1 which would mean no volume change. Looking at the absolute difference from 1 allows for extreme contraction or expansion to be considered, although there are reasons this is certainly non linear in any case. This is also related to the density ratio, so it is not surprising to see it appear.
"L/T": geometry ratios showing up also makes sense - if a material is long and thin it will have an easier time bending and twisting to relieve internal strain due to the transformation. However if a material is blocky and chunky, it will have a much harder time relieving that strain, and may break due to that buildup.
"ref beta": The transformed structural parameter beta showing up as a feature with importance is surprising for the reasons explained below when discussing other structural parameters as least important.
"wavelength nm": This is also surprising result and may have to do with the data set. The wavelengths of this dataset are mostly in UV close together, but there are a few that are extremely different, so I'm cautious to believe the relevance of wavelength from this dataset.

The least important features:

"ref gamma" and "ref c": These are both structural parameters of the initial phase - it would be expected that they don't have high importance because it doesn't matter what they are individually, but how they relate to one another via the strain inducing deformation.
"Stretch 23": The fact that the 12 and 23 positions in the stretch tensor have little importance could mean that the shear components of deformation/stretch won't impact the reversibility.
"Ratio of symmetry" and "Z'" ratio: these are both a bit related to the symmetry change that may occur through a phase transformation. This could potentially help accomodate strain in the material, however, in our data set most data points did not undergo any symmetry change, so it is not surprising this did not have an impact.

### Feature analysis for gradient boosting model

In [ ]:
gbc_importances = gbc.feature_importances_

fig, ax = plt.subplots()
ax.bar(range(len(gbc_importances)), gbc_importances, color='blue')
ax.set_xlabel('Feature Index Number')
ax.set_ylabel('Normalized Importance')
ax.grid()

In [ ]:
#find most impacting features
list_gbc_i = gbc_importances.tolist()
rank_importances_gbc = np.argsort(list_gbc_i)

#find n most importance features
n=5
print('Most important features to predict reversiblity (highest first):\n')

for i in range(n):
    print(f"{i+1}. {feature_names[rank_importances_gbc[feature_number-1-i]]}")

In [ ]:
#find least impacting features

#find m most importance features
m=5
print('Least important features to predict reversiblity (lowest first):\n')

for i in range(m):
    print(f"{feature_number-i}. {feature_names[rank_importances_gbc[i]]}")

#### Discussion - How does the gradient boosting feature importance compare to the random forest? Do they agree or contradict?
The feature importances make less sense for the gradient boosted model, which may be related to the low scoring for this model.

Again the geometry parameter is most important. The structural parameters listed are harder to rationalize as important with physical intuition. Lambda 2 as the second principle stretch could make sense as it is related to the compatibility criteria in martensitic transformations. The 23 stretch tensor value feels like it may have more to do with the dataset, although it is hard to say.

The least important features as structure parameters makes some sense. The Z' ratio showing up again alligns with the random forest. Lambda 3 is the third principle stretch value, so it also is not surprising. The Stretch 33 value is most surprising as it relates to expansion or contraction which the importance of density and volume change in the other model seems to contradict.

These feature importances seem a bit less likely, but this is a very small data set in any case, so all interpretations are speculative.

### Attempt this problem with other models:

### Logistic Regression:

In [ ]:
#initialize logistic regression:
lr = LogisticRegression()
lr.fit(X_train, y_train)
print(f"Logistic regression training score: {lr.score(X_train, y_train)}")
print(f"Logistic regression test score: {lr.score(X_test, y_test)}")

This looks like a very poor performing model, not worth analyzing coefficients of features.

### Ridge Classifier:

In [ ]:
#Must scale data to use Ridge Classifier in order to be able to analyze coefficient meaning
#to find the best alpha for Ridge Classifier, use cross validation on only the training set
#this way, the best alpha can be included in the model that will be tested on test data
best_score = 0
best_alpha = None

#iterate over log scale range for alpha hyperparameter
for alpha in [0.001, 0.01, 0.1, 1, 10, 100]:
    #Create pipeline to scale without leaking
    rc_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('rc', RidgeClassifier(alpha=alpha))
    ])
    
    #cross validation with given alpha using pipeline
    scores = cross_val_score(rc_pipe, X_train, y_train, cv=5)
    score = np.mean(scores)
    #print(score)
    
    #if improves score, save it
    if score > best_score:
         best_score = score
         best_alpha = alpha
         
print(f"Best cross validated score: {best_score}")
print(f"Best alpha when best cross validated score: {best_alpha}")

In [ ]:
#initialize Ridge classifier with best alpha (ba) using pipeline fore scaling
rc_ba = Pipeline([
    ('scaler', StandardScaler()),
    ('rc', RidgeClassifier(alpha=best_alpha))
])
rc_ba.fit(X_train, y_train)
print(f"Ridge classifier training score: {rc_ba.score(X_train, y_train)}")
print(f"Ridge classifier test score: {rc_ba.score(X_test, y_test)}")

In [ ]:
#cross validation now using the whole dataset with test set and found best alpha
rc_cv_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('rc', RidgeClassifier(alpha=best_alpha))
])
rc_scores = cross_val_score(rc_cv_pipe, X, y, cv=5)
print(f"Ridge classifier cross validation scores: {rc_scores}")
print(f"Ridge classifier average cross validation scores: {rc_scores.mean()}")

### Feature analysis for Ridge Classifier

In [ ]:
rc_coef = rc_ba.named_steps['rc'].coef_

fig, ax = plt.subplots()
ax.bar(range(len(rc_coef)), rc_coef, color='blue')
ax.set_xlabel('Feature Index Number')
ax.set_ylabel('Coefficient Weight')
ax.grid()

In [ ]:
rc_abscoef = np.abs(rc_coef)
rank_coef_rc = np.argsort(rc_abscoef)

#find n most importance features
n=5
print('Most important features to predict reversiblity (highest first):\n')

for i in range(n):
    print(f"{i+1}. {feature_names[rank_coef_rc[feature_number-1-i]]}")


In [ ]:
#find least impacting features

#find m most importance features
m=5
print('Least important features to predict reversiblity (lowest first):\n')

for i in range(m):
    print(f"{feature_number-i}. {feature_names[rank_coef_rc[i]]}")

#### Discussion - How does the ridge classifier compare to the forest models?
This feature importance alligns closely with that found from the random forest. The only difference is the inclusion of another geometry ratio and a lack of the questionable wavelength inclusion.

The occurence of "ref beta" as an important feature in every model is surprising. It likeley has more to do with the dataset than some energy function that dictates reversibility, which is what we are hoping to create. There are many monoclinic molecular crystals, so there is a lot of variance of beta in our data set. On the other hand, the severity of beta in the transformed state could be an unfavorable orientation that is difficult to relax via reversible motion leading to irreversibility in the transformed state.

Similarly, the least important features are the structure parameters, and the smallest principle stretch, all of which intuitively make sense to be less important.